In [47]:
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, StratifiedKFold

In [48]:
df = pd.read_csv("modeling_ready_dataset.csv")

key_cols = ["season", "team"]
target_col = "made_playoffs"
feature_cols = [c for c in df.columns if c not in key_cols + [target_col]]

Creating 5 expanding window folds for temporal validation. 

Each fold trains on all prior seasons and tests on the next unseen year, ensuring the model never sees future data during training. We want to assess performance stability across multiple seasons

In [49]:
folds = []
for test_year in range(2021, 2026):
    train_mask = df["season"] < test_year
    test_mask = df["season"] == test_year

    fold = {
        "fold_num": test_year - 2020,
        "test_year": test_year,
        "train_seasons": sorted(df[train_mask]["season"].unique()),
        "X_train": df.loc[train_mask, feature_cols],
        "y_train": df.loc[train_mask, target_col],
        "X_test": df.loc[test_mask, feature_cols],
        "y_test": df.loc[test_mask, target_col],
    }
    folds.append(fold)

In [50]:
print(f"{'Fold':<6s} {'Train Seasons':<10s} {'Train Rows':>10s} {'Train PO%':>10s} {'Test Year':>10s} {'Test Rows':>10s} {'Test PO%':>9s}")

for fold in folds:
    train_seasons_str = f"2018-{fold['train_seasons'][-1]}"
    train_po = fold["y_train"].mean() * 100
    test_po = fold["y_test"].mean() * 100
    print(f"{fold['fold_num']:<6d} {train_seasons_str:<10s} {len(fold['X_train']):>10d} {train_po:>9.1f}% {fold['test_year']:>10d} {len(fold['X_test']):>10d} {test_po:>8.1f}%")

Fold   Train Seasons Train Rows  Train PO%  Test Year  Test Rows  Test PO%
1      2018-2020          96      39.6%       2021         32     43.8%
2      2018-2021         128      40.6%       2022         32     43.8%
3      2018-2022         160      41.2%       2023         32     43.8%
4      2018-2023         192      41.7%       2024         32     43.8%
5      2018-2024         224      42.0%       2025         32     43.8%


Verifying that no data leakage exists by confirming each fold's test year comes strictly after its latest training season. If any test data overlapped with training data, the assertion would stop the script and flag the problem

In [51]:
# Verify no data leakage ---
print("Data leakage verification:")
for fold in folds:
    max_train = fold["train_seasons"][-1]
    assert fold["test_year"] > max_train, f"LEAKAGE in fold {fold['fold_num']}!"
    print(f"  Fold {fold['fold_num']}: Train through {max_train}, test on {fold['test_year']} — OK")

Data leakage verification:
  Fold 1: Train through 2020, test on 2021 — OK
  Fold 2: Train through 2021, test on 2022 — OK
  Fold 3: Train through 2022, test on 2023 — OK
  Fold 4: Train through 2023, test on 2024 — OK
  Fold 5: Train through 2024, test on 2025 — OK


### Baseline Benchmarks

Before building any real models, we establish baseline benchmarks to measure against. The evaluate_model helper function computes accuracy, precision, recall, F1, and AUC-ROC so we can reuse it consistently across all models. After creating the helper function, the first baseline always predicts non-playoff (0) for every team, representing the simplest possible approach. Any useful model must outperform this floor. Results are stored in all_results so we can compare all models side by side later.

In [52]:
# Helper function to compute all metrics
def evaluate_model(y_true, y_pred, y_prob=None):
    """Compute standard classification metrics."""
    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
    }
    if y_prob is not None:
        metrics["auc_roc"] = roc_auc_score(y_true, y_prob)
    return metrics

In [53]:
# Storage for all results
all_results = []

# --- Baseline 1: Naive Majority Class ---
print("\n--- Baseline 1: Naive Majority Class (always predict 0) ---")
print(f"  {'Fold':<6s} {'Test Year':>9s} {'Acc':>7s} {'Prec':>7s} {'Recall':>7s} {'F1':>7s}")
print(f"  {'-'*40}")

for fold in folds:
    y_pred = np.zeros(len(fold["y_test"]), dtype=int)
    metrics = evaluate_model(fold["y_test"], y_pred)
    metrics["auc_roc"] = 0.5

    print(f"  {fold['fold_num']:<6d} {fold['test_year']:>9d} {metrics['accuracy']:>7.3f} {metrics['precision']:>7.3f} {metrics['recall']:>7.3f} {metrics['f1']:>7.3f}")

    all_results.append({
        "model": "Naive Majority",
        "fold": fold["fold_num"],
        "test_year": fold["test_year"],
        **metrics
    })


--- Baseline 1: Naive Majority Class (always predict 0) ---
  Fold   Test Year     Acc    Prec  Recall      F1
  ----------------------------------------
  1           2021   0.562   0.000   0.000   0.000
  2           2022   0.562   0.000   0.000   0.000
  3           2023   0.562   0.000   0.000   0.000
  4           2024   0.562   0.000   0.000   0.000
  5           2025   0.562   0.000   0.000   0.000


Observing the output we see it predicts 56.2% accuracy just by saying no one will make the playoffs. Precision, recall, and F1 are all 0 because the model never predicts a single team as a playoff team. We gather that accuracy alone can be misleading because 56% sounds decent, but its useless when it cant identify a single playoff team. This baseline sets the absolute floor, so any model must beat the accuracy and also actually identify playoff teams.

**Second Baseline**

Our second baseline tests how far a single feature can go without any machine learning. Using score_pct_diff, which was our strongest predictor at r=0.704, we find the optimal threshold on training data that maximizes F1, then apply it to the test set.  If a team's scoring efficiency differential is above the threshold we want to predict playoff, otherwise, we predict non-playoffs.

In [55]:
print(f"  {'Fold':<6s} {'Test Year':>9s} {'Threshold':>10s} {'Acc':>7s} {'Prec':>7s} {'Recall':>7s} {'F1':>7s} {'AUC':>7s}")
print(f"  {'-'*60}")
best_feature = "score_pct_diff"

for fold in folds:
    train_vals = fold["X_train"][best_feature]
    train_labels = fold["y_train"]

    best_threshold = 0
    best_f1 = 0

    for threshold in np.arange(train_vals.min(), train_vals.max(), 0.5):
        preds = (train_vals >= threshold).astype(int)
        f1 = f1_score(train_labels, preds, zero_division=0)
        if f1 > best_f1:
            best_f1 = f1
            best_threshold = threshold

    y_pred = (fold["X_test"][best_feature] >= best_threshold).astype(int)

    train_min = train_vals.min()
    train_max = train_vals.max()
    y_prob = (fold["X_test"][best_feature] - train_min) / (train_max - train_min)
    y_prob = y_prob.clip(0, 1)

    metrics = evaluate_model(fold["y_test"], y_pred, y_prob)

    print(f"  {fold['fold_num']:<6d} {fold['test_year']:>9d} {best_threshold:>10.1f} {metrics['accuracy']:>7.3f} {metrics['precision']:>7.3f} {metrics['recall']:>7.3f} {metrics['f1']:>7.3f} {metrics['auc_roc']:>7.3f}")

    all_results.append({
        "model": "Heuristic (score_pct_diff)",
        "fold": fold["fold_num"],
        "test_year": fold["test_year"],
        **metrics
    })
 

  Fold   Test Year  Threshold     Acc    Prec  Recall      F1     AUC
  ------------------------------------------------------------
  1           2021       -1.2   0.750   0.650   0.929   0.765   0.929
  2           2022       -0.5   0.781   0.733   0.786   0.759   0.881
  3           2023       -1.0   0.875   0.812   0.929   0.867   0.909
  4           2024       -1.0   0.844   0.765   0.929   0.839   0.972
  5           2025       -1.0   0.750   0.667   0.857   0.750   0.875


In [68]:
# Producing the averages 
heuristic = results_df[results_df["model"] == "Heuristic (score_pct_diff)"]
for metric in ["accuracy", "precision", "recall", "f1", "auc_roc"]:
    print(round(heuristic[metric].mean(), 3))

0.8
0.725
0.886
0.796
0.913


Observing this output we can takeaway that a single feature with a simple threshold achieves 80% accuracy and .796 average F1 across all folds. A high recall average of 88.6% means it catches most playoff teams. While the precison score of 72.5% means it also flags non-playoff teams incorrectly. 

So, with our upcoming models of Logistic Regression, Random Forest and Gradient Boosting we need to beat the 0.796 F1 to justify their use over this simple one feature rule.

### Logistic Regression

We want to determine of the 64 features we still have if we should eliminate the weak features, or keep them all and how aggressively to regularize the data. Using the 2 penalties L1(Lasso), and L2(Ridge) will help determine feature results, while C will control how much regularization is applied.

In [71]:
param_grid_lr = {
    "C": [0.001, 0.01, 0.1, 1.0, 10.0],
    "penalty": ["l1", "l2"],
}


In [ ]:
lr_models = []

for fold in folds:
    # Scale features: fit on training data ONLY to prevent leakage
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(fold["X_train"])
    X_test_scaled = scaler.transform(fold["X_test"])

    cv_inner = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    lr = LogisticRegression(solver="saga", max_iter=5000, random_state=42)

    # Find the best C and penalty combination out of our 10 options from param_grid_lr
    grid_search = GridSearchCV(
        estimator=lr,
        param_grid=param_grid_lr,
        cv=cv_inner,
        scoring="f1",
        n_jobs=-1,
        refit=True,
    )

    grid_search.fit(X_train_scaled, fold["y_train"])
    
    best_lr = grid_search.best_estimator_
    best_params = grid_search.best_params_

    y_pred = best_lr.predict(X_test_scaled)
    y_prob = best_lr.predict_proba(X_test_scaled)[:, 1]

    metrics = evaluate_model(fold["y_test"], y_pred, y_prob)

    print(f"  {fold['fold_num']:<6d} {fold['test_year']:>9d} {best_params['C']:>7.3f} {best_params['penalty']:>5s} "
          f"{metrics['accuracy']:>7.3f} {metrics['precision']:>7.3f} {metrics['recall']:>7.3f} {metrics['f1']:>7.3f} {metrics['auc_roc']:>7.3f}")

    all_results.append({
        "model": "Logistic Regression",
        "fold": fold["fold_num"],
        "test_year": fold["test_year"],
        **metrics
    })

    lr_models.append({
        "fold": fold["fold_num"],
        "test_year": fold["test_year"],
        "model": best_lr,
        "scaler": scaler,
        "best_params": best_params,
        "cv_best_score": grid_search.best_score_,
    })

  1           2021   1.000    l1   0.812   0.750   0.857   0.800   0.905
  2           2022   0.100    l2   0.812   1.000   0.571   0.727   0.968
  3           2023  10.000    l1   0.969   1.000   0.929   0.963   0.992
  4           2024  10.000    l1   1.000   1.000   1.000   1.000   1.000
  5           2025  10.000    l1   0.938   0.929   0.929   0.929   0.984
